Importing

In [ ]:
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


Import Kaggle dataset

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Download latest version
path = kagglehub.dataset_download("jamaltariqcheema/pima-indians-diabetes-dataset")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "jamaltariqcheema/pima-indians-diabetes-dataset",
  'diabetes.csv',
)
print("First 5 records:", df.head())

Using Colab cache for faster access to the 'pima-indians-diabetes-dataset' dataset.
Path to dataset files: /kaggle/input/pima-indians-diabetes-dataset
['diabetes.csv']


/tmp/ipykernel_258/1255632500.py:12: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'pima-indians-diabetes-dataset' dataset.
First 5 records:    Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148           72.0             35    169.5  33.6   
1            1       85           66.0             29    102.5  26.6   
2            8      183           64.0             32    169.5  23.3   
3            1       89           66.0             23     94.0  28.1   
4            0      137           40.0             35    168.0  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


remove missing/irrelevant information

In [ ]:
# Glucose, BloodPressure, SkinThickness, Insulin, BMI can't be zero
can_not_be_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[can_not_be_zero] = df[can_not_be_zero].replace(0, np.nan)

#Optional: remove rows with too many missing values
df = df.dropna(thresh=len(df.columns) // 2 + 1)

#fill missing values with median
for col in can_not_be_zero:
    df[col] = df[col].fillna(df[col].median())
# Optional: remove extreme insulin values
df['Insulin'] = df['Insulin'].clip(upper=500)

#remove duplicates
df.drop_duplicates(inplace=True)


erroneous value

In [ ]:
# number of unusual values
print((df['Pregnancies'] > 15).sum())
print(((df['Glucose'] < 40) | (df['Glucose'] > 250)).sum())
print(((df['BloodPressure'] < 40) | (df['BloodPressure'] > 130)).sum())
print((df['SkinThickness'] > 80).sum())
print((df['Insulin'] > 500).sum())
print((df['BMI'] < 10 | (df['BMI'] > 60)).sum())
print((df['DiabetesPedigreeFunction'] > 2.5).sum())
print((df['Age'] > 100).sum())

#check what those values are
print(df[df['Pregnancies'] > 15])
print(df[(df['BloodPressure'] < 40) | (df['BloodPressure'] > 130)])
print(df[df['SkinThickness'] > 80])
#suspicious, some blood pressures are like 24, 30, 38
#capping (I recommend that):
df['BloodPressure'] = df['BloodPressure'].clip(lower=40, upper=130)
df['SkinThickness'] = df['SkinThickness'].clip(upper=80)
#or removing:
#df = df[df['BloodPressure'] >= 30]
#df = df[df['SkinThickness'] < 80]


1
0
0
0
0
0
0
0
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
159           17      163           72.0             41    114.0  40.9   

     DiabetesPedigreeFunction  Age  Outcome  
159                     0.817   47        1  
Empty DataFrame
Columns: [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, Outcome]
Index: []
Empty DataFrame
Columns: [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, Outcome]
Index: []


normalize

In [ ]:
# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
import pandas as pd
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Combine back
df_scaled = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)